# Samsung Dissatisfaction Risk Prediction Model

In this notebook, we build a machine learning model to predict whether a Samsung customer review shows high dissatisfaction risk.

The target variable is `dissatisfaction_risk`.

- 1 means High Dissatisfaction Risk
- 0 means Low Dissatisfaction Risk

To avoid data leakage, we do not use `rating` or `sentiment` as input features because they were already used to create the target.

In [1]:
# Environment: VS Code Jupyter Notebook
# Notebook 02: Prediction Model

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Environment: VS Code Jupyter Notebook
# Load the Samsung modelling dataset

data_path = Path("../data_cleaned/samsung_modeling_dataset.csv")

modeling_df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows and columns:", modeling_df.shape)

modeling_df.head()

Dataset loaded successfully.
Rows and columns: (7052, 16)


,age,price_usd,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,review_length,word_count,helpful_votes,country,language,source,model,dissatisfaction_risk
0,39,1146.49,True,2,3,1,2,3,68,14,4,UK,English,BestBuy,Galaxy Note 20,0
1,30,1369.88,True,3,3,3,5,3,50,8,5,India,Hindi,BestBuy,Galaxy S24,0
2,45,1071.57,True,4,5,5,4,2,64,12,4,Canada,English,Flipkart,Galaxy A55,0
3,38,1147.47,True,1,2,2,1,2,50,8,4,India,Hindi,eBay,Galaxy S24,1
4,29,924.51,False,4,4,3,3,4,78,14,5,USA,English,Flipkart,Galaxy S24,0


In [3]:
# Environment: VS Code Jupyter Notebook
# Check dissatisfaction risk distribution

risk_distribution = modeling_df["dissatisfaction_risk"].value_counts()
risk_percentage = modeling_df["dissatisfaction_risk"].value_counts(normalize=True) * 100

print("Risk count:")
print(risk_distribution)

print("\nRisk percentage:")
print(risk_percentage.round(2))

Risk count:
dissatisfaction_risk
0    4627
1    2425
Name: count, dtype: int64

Risk percentage:
dissatisfaction_risk
0    65.61
1    34.39
Name: proportion, dtype: float64


In [4]:
# Environment: VS Code Jupyter Notebook
# Check all columns available for modelling

modeling_df.columns

Index(['age', 'price_usd', 'verified_purchase', 'battery_life_rating',
       'camera_rating', 'performance_rating', 'design_rating',
       'display_rating', 'review_length', 'word_count', 'helpful_votes',
       'country', 'language', 'source', 'model', 'dissatisfaction_risk'],
      dtype='str')

In [5]:
# Environment: VS Code Jupyter Notebook
# Separate input features X and target variable y

X = modeling_df.drop(columns=["dissatisfaction_risk"])
y = modeling_df["dissatisfaction_risk"]

print("Input features shape:", X.shape)
print("Target shape:", y.shape)

Input features shape: (7052, 15)
Target shape: (7052,)


In [6]:
# Environment: VS Code Jupyter Notebook
# Identify numerical and categorical columns

numeric_features = [
    "age",
    "price_usd",
    "battery_life_rating",
    "camera_rating",
    "performance_rating",
    "design_rating",
    "display_rating",
    "review_length",
    "word_count",
    "helpful_votes"
]

categorical_features = [
    "verified_purchase",
    "country",
    "language",
    "source",
    "model"
]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'price_usd', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'review_length', 'word_count', 'helpful_votes']
Categorical features: ['verified_purchase', 'country', 'language', 'source', 'model']


In [7]:
# Environment: VS Code Jupyter Notebook
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (5641, 15)
Testing features: (1411, 15)
Training target: (5641,)
Testing target: (1411,)


In [8]:
# Environment: VS Code Jupyter Notebook
# Build preprocessing and logistic regression model pipeline

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

In [9]:
# Environment: VS Code Jupyter Notebook
# Train the prediction model

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [10]:
# Environment: VS Code Jupyter Notebook
# Make predictions on the test set

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Predictions created successfully.")

Predictions created successfully.


In [11]:
# Environment: VS Code Jupyter Notebook
# Evaluate the model performance

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("Accuracy:", round(accuracy, 3))
print("ROC AUC:", round(roc_auc, 3))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.906
ROC AUC: 0.975

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.90      0.93       926
           1       0.83      0.91      0.87       485

    accuracy                           0.91      1411
   macro avg       0.89      0.91      0.90      1411
weighted avg       0.91      0.91      0.91      1411



In [12]:
# Environment: VS Code Jupyter Notebook
# Create a confusion matrix

conf_matrix = confusion_matrix(y_test, y_pred)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Actual Low Risk", "Actual High Risk"],
    columns=["Predicted Low Risk", "Predicted High Risk"]
)

conf_matrix_df

,Predicted Low Risk,Predicted High Risk
Actual Low Risk,835,91
Actual High Risk,42,443


In [13]:
# Environment: VS Code Jupyter Notebook
# Create a results table with actual values, predictions, and risk probability

prediction_results = X_test.copy()

prediction_results["actual_dissatisfaction_risk"] = y_test.values
prediction_results["predicted_dissatisfaction_risk"] = y_pred
prediction_results["predicted_high_risk_probability"] = y_pred_proba

prediction_results["actual_risk_label"] = prediction_results["actual_dissatisfaction_risk"].map({
    1: "High Dissatisfaction Risk",
    0: "Low Dissatisfaction Risk"
})

prediction_results["predicted_risk_label"] = prediction_results["predicted_dissatisfaction_risk"].map({
    1: "High Dissatisfaction Risk",
    0: "Low Dissatisfaction Risk"
})

prediction_results.head()

,age,price_usd,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,review_length,word_count,helpful_votes,country,language,source,model,actual_dissatisfaction_risk,predicted_dissatisfaction_risk,predicted_high_risk_probability,actual_risk_label,predicted_risk_label
5291,46,852.80,True,3,3,3,3,2,51,9,5,USA,English,Amazon,Galaxy A55,0,0,0.092662,Low Dissatisfaction Risk,Low Dissatisfaction Risk
5468,30,455.35,True,1,2,1,1,1,61,8,2,USA,English,BestBuy,Galaxy A55,1,1,0.996262,High Dissatisfaction Risk,High Dissatisfaction Risk
1836,35,1103.76,True,3,4,4,4,3,71,12,1,UK,English,eBay,Galaxy A55,0,0,0.004429,Low Dissatisfaction Risk,Low Dissatisfaction Risk
2843,31,1297.79,True,1,1,4,3,1,69,13,4,Brazil,Portuguese,Flipkart,Galaxy A55,0,1,0.570638,Low Dissatisfaction Risk,High Dissatisfaction Risk
5992,40,910.25,True,2,1,1,1,1,61,8,2,UAE,English,Flipkart,Galaxy Note 20,1,1,0.995200,High Dissatisfaction Risk,High Dissatisfaction Risk


In [14]:
# Environment: VS Code Jupyter Notebook
# Create a simple model metrics table

model_metrics = pd.DataFrame({
    "metric": ["Accuracy", "ROC AUC"],
    "score": [accuracy, roc_auc]
})

model_metrics["score"] = model_metrics["score"].round(3)

model_metrics

,metric,score
0,Accuracy,0.906
1,ROC AUC,0.975


In [15]:
# Environment: VS Code Jupyter Notebook
# Prepare confusion matrix for saving

conf_matrix_export = conf_matrix_df.reset_index().rename(columns={
    "index": "actual_class"
})

conf_matrix_export

,actual_class,Predicted Low Risk,Predicted High Risk
0,Actual Low Risk,835,91
1,Actual High Risk,42,443


In [16]:
# Environment: VS Code Jupyter Notebook
# Extract feature importance from the logistic regression model

preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()
coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

feature_importance["absolute_importance"] = feature_importance["coefficient"].abs()

feature_importance = feature_importance.sort_values(
    by="absolute_importance",
    ascending=False
)

feature_importance.head(20)

,feature,coefficient,absolute_importance
3,num__camera_rating,-1.227876,1.227876
6,num__display_rating,-1.174845,1.174845
2,num__battery_life_rating,-1.150083,1.150083
4,num__performance_rating,-1.055432,1.055432
9,num__helpful_votes,-1.041613,1.041613
5,num__design_rating,-1.004495,1.004495
10,cat__verified_purchase_False,-0.502058,0.502058
8,num__word_count,-0.473114,0.473114
31,cat__model_Galaxy S24,-0.440905,0.440905
11,cat__verified_purchase_True,-0.413986,0.413986


In [17]:
# Environment: VS Code Jupyter Notebook
# Save model outputs for Power BI and reporting

output_folder = Path("../data_cleaned")
output_folder.mkdir(exist_ok=True)

prediction_results.to_csv(output_folder / "samsung_prediction_results.csv", index=False)
model_metrics.to_csv(output_folder / "samsung_model_metrics.csv", index=False)
conf_matrix_export.to_csv(output_folder / "samsung_confusion_matrix.csv", index=False)
feature_importance.to_csv(output_folder / "samsung_model_feature_importance.csv", index=False)

print("Model output files saved successfully.")

Model output files saved successfully.


# Model Interpretation

The logistic regression model achieved strong performance in predicting Samsung customer dissatisfaction risk.

The model reached an accuracy of 90.6% and a ROC AUC of 97.5%, showing that it can strongly separate low-risk reviews from high-risk reviews.

From the confusion matrix, the model correctly identified 443 high-risk reviews out of 485 actual high-risk cases in the test set. This means the model is effective at detecting risky customer experiences.

In a business context, this type of model could help Samsung automatically prioritize customer feedback that may indicate trust, satisfaction, or future adoption problems.

The model should be interpreted as a customer feedback intelligence model, not as a perfect future forecasting system. It uses review-based signals and product experience ratings to estimate dissatisfaction risk.